In [51]:
import sys # for automation and parallelization: set manual to false when run by a launcher
import json
MANUAL, SCENARIO = (True, 'base') if 'ipykernel' in sys.argv[0]\
     else (False, json.loads(sys.argv[1])['scenario'])

# START

In [52]:
import sys
sys.path.insert(0, r'../../../quetzal')

from quetzal.model import stepmodel, connectionscanmodel
from quetzal.io import excel
import geopandas as gpd
import shapely
import os
import numpy as np
import pandas as pd
if MANUAL:
    %matplotlib inline

from syspy.syspy_utils.data_visualization import bandwidth, width_series, color_series

In [53]:
qpath = r'../../'
ipath = qpath + r'inputs/'
mpath = qpath + r'scenarios/{}/models/'.format(SCENARIO)

In [54]:
var = excel.read_var(ipath + r'parameters.xlsx', scenario=SCENARIO)

Index(['description', 'unit', 'category', 'parameter', 'base',
       'base_timetables'],
      dtype='object')


In [55]:
INTRA_AREA_TO_DISTANCE_FACTOR = var[('step_distribution', 'intra_area_to_distance_factor')]

WALK_TIME_FACTOR = var[('mode_penalty', 'walk_time_factor')]
BIKE_TIME_FACTOR = var[('mode_penalty', 'bike_time_factor')]
CAR_TIME_FACTOR = var[('mode_penalty', 'car_time_factor')]

WAITING_TIME_FACTOR = var[('pt_penalty', 'waiting_time_factor')]
TRANSFER_TIME_FACTOR = var[('pt_penalty', 'transfer_time_factor')]
ACCESS_TIME_FACTOR = var[('pt_penalty', 'access_time_factor')]
TRANSFER_PENALTY = var[('pt_penalty', 'transfer_penalty')]

VALUE_OF_TIME_SEC = var[('value_of_time', 'value_of_time_h')]/3600

In [56]:
if var['run_mode']['mode'] != 'timetables':
    sm = stepmodel.read_zippedpickles(mpath + r'generation_distribution')
else:
    sm = connectionscanmodel.read_zippedpickles(mpath + r'generation_distribution')

zone_to_road: 100%|██████████| 22/22 [00:01<00:00, 13.55it/s]


## MaP

In [57]:
sm.los = sm.los[sm.los["route_type"]!="walk"]
sm.segments = ["volume"]

In [58]:
sm.los.loc[(sm.los['origin'] == 'zone_05') & (sm.los['destination'] == 'zone_04')][['route_types'] + [x for x in sm.los.columns if 'time' in x] + ['length', 'fare']]

,route_types,time,gtime,time_link_path,access_time,footpath_time,in_vehicle_time,waiting_time,boarding_time,length,fare
339,"(car,)",3650.324355,10274.262356,NaN,NaN,NaN,NaN,NaN,NaN,44206.116502,9.393800
4895,"(bike,)",10274.262356,10133.261954,NaN,NaN,NaN,NaN,NaN,NaN,42252.877772,0.000000
9451,"(bus,)",19500.713630,22958.996847,8378.283218,1563.647494,2217.066136,4920.0,10800.0,0.0,NaN,6.806767
14007,"(bike,)",8798.088906,9994.942222,0.000000,360.711724,8437.377182,0.0,0.0,0.0,NaN,0.000000


In [59]:
### Compute zone-zone distances

sm.zones['area'] = sm.zones.area ## m²

distances = sm.zones.reset_index()[['index', 'lat_centroid', 'lng_centroid', 'area']]
distances['geometry'] = gpd.points_from_xy(distances['lng_centroid'], distances['lat_centroid'])
distances = distances.set_geometry('geometry').set_crs('EPSG:4326').to_crs('EPSG:2154').drop(columns=['lng_centroid', 'lat_centroid'])
distances = distances.merge(distances, how='cross', suffixes = ['_from', '_to'])
distances['distance'] = (distances['area_from'] ** 0.5) * INTRA_AREA_TO_DISTANCE_FACTOR
distances.loc[distances['index_from'] != distances['index_to'], 'distance'] = distances.apply(lambda x: shapely.distance(x['geometry_from'], x['geometry_to']), 1)
distances = distances[['index_from', 'index_to', 'distance']].rename(columns={'index_from': 'origin', 'index_to': 'destination'})

In [60]:
### Share of walk among volumes:
share_not_walking = {
    1000: 0.12,
    2000: 0.41,
    3000: 0.87,
    4000: 0.97,
    5000: 0.98, 
    np.inf: 1
}

def apply_dict(x, dict):
    for v_ in dict.keys():
        if x <= v_:
            return dict[v_]

sm.volumes = sm.volumes.merge(distances, on=['origin', 'destination'], how='left')
sm.volumes['volume'] *= sm.volumes['distance'].apply(lambda x: apply_dict(x, share_not_walking))
sm.volumes.drop(columns='distance', inplace=True)

## Configure Logit scales

In [61]:
var['preparation_logit'].to_dict()

{'time': -0.00033860092781289584,
 'price': -0.000334658514016994,
 'transfers': -0.5,
 'mode': 1,
 'pt_mode': 0.8,
 'pt_path': 0.5}

In [62]:
kwargs = var['preparation_logit'].to_dict()

sm.preparation_logit(
    **kwargs, segments=['volume'],
    route_types=['root', 'car', 'bike', 'pt'] + list(sm.links["route_type"].unique())
)

In [63]:
sm.utility_values.T

value,time,price,ntransfers,mode_utility
segment,,,,
root,-0.000339,-0.000335,-0.5,1.0
volume,-0.000339,-0.000335,-0.5,1.0


In [64]:
sm.logit_scales.T

route_type,root,car,bike,pt,bus,rail,tramway
segment,,,,,,,
root,1.0,0.0,0.5,0.8,0.5,0.5,0.5
volume,1.0,0.0,0.5,0.8,0.5,0.5,0.5


In [65]:
sm.mode_nests.loc['bike'] = 'root'
sm.mode_nests

segment,root,volume
route_type,,
root,NaN,NaN
car,root,root
bike,root,root
pt,root,root
bus,pt,pt
rail,pt,pt
tramway,pt,pt


In [66]:
# TODO: ajouter une valeur par défaut pour les paramètres de mode TC manquants ?
for mode in set(sm.mode_utility.index) - {"root", "pt"}:
    sm.mode_utility.loc[mode] = var['mode_utility'][mode]

sm.mode_utility

segment,root,volume
route_type,,
root,0,0
car,1,1
bike,0,0
pt,0,0
bus,0,0
rail,0,0
tramway,0,0


## Utility

In [67]:
sm.los['price'] = sm.los['fare']
# TODO Louison: (temps de transfer? = temps d'attente pas lié au premier moyen de transport utilisé?)
# temps de rabattement : access_time
# temps de correspondance : waiting_time
# temps d'accès : footpath_time
### et finalement : time = in_vehicle_time + access_time + footpath_time + waiting_time

In [68]:
pt_mask = sm.los['route_type'].isin(['rail', 'tramway', 'bus'])
car_mask = (sm.los['route_type'] == 'car')
bike_mask = (sm.los['route_type'] == 'bike')
walk_mask = (sm.los['route_type'] == 'walk')

sm.los.loc[pt_mask, 'gtime'] = sm.los.loc[pt_mask, 'in_vehicle_time'].fillna(0) +\
    sm.los.loc[pt_mask, 'access_time'].fillna(0)*ACCESS_TIME_FACTOR +\
    sm.los.loc[pt_mask, 'footpath_time'].fillna(0)*ACCESS_TIME_FACTOR +\
    sm.los.loc[pt_mask, 'waiting_time'].fillna(0)*WAITING_TIME_FACTOR +\
    sm.los.loc[pt_mask, 'price'].fillna(0)/VALUE_OF_TIME_SEC +\
    sm.los.loc[pt_mask, 'ntransfers']*TRANSFER_PENALTY

sm.los.loc[car_mask, 'gtime'] = sm.los.loc[car_mask, 'time']*CAR_TIME_FACTOR +\
    sm.los.loc[car_mask, 'fare']/VALUE_OF_TIME_SEC

sm.los.loc[bike_mask, 'gtime'] = sm.los['time']*BIKE_TIME_FACTOR
sm.los.loc[walk_mask, 'gtime'] = sm.los['time']*WALK_TIME_FACTOR

# TODO : BOUCLE à crée avec les cellules dessous, à mettre au propre dans des fonctions avant

Principe: affectation par tranche et mise à jour des LOS
on découpe la demande en 10 (paramétrable) tranches et on fait 10 itérations:

- a. Logit
- b. on fait l’affectation TC et on met à jour les LOS TC
- c. on fait l’affectation CAR (AON) et on met à jour les LOS CAR
- d. on fusionne tous les LOS mis à jour et on passe à la tranche suivante

Les tranches sont de taille différente (plus grosses au début)

## A. RUN LOGIT

In [69]:
mode_capacities = {
    'tramway': 250,
    'bus': 50,
    'rail': 600
}

sm.links['mode_capacity'] = sm.links['route_type'].map(mode_capacities)
sm.links['capacity'] = 3600*sm.links['mode_capacity']/sm.links['headway']

In [70]:
# Calculate the utilities and the probabilities
sm.analysis_mode_utility(how='mean')
sm.initialize_logit()
sm.step_logit()

100%|██████████| 10/10 [00:00<00:00, 25.93it/s]


# B. ASSIGNMENT

### Road assignment

In [71]:
bike = sm.copy()
bike.car_los = bike.los.loc[bike.los['route_type'] == 'bike']
bike.step_car_assignment()
bike.road_links = bike.road_links.drop(columns=[('volume', 'car')]).rename(columns={('all', 'car'): 'volume_bike'})

volume: 100%|██████████| 1/1 [00:00<00:00,  1.79it/s]


In [72]:
car = sm.copy()
car.car_los = car.los.loc[car.los['route_type'] == 'car']
car.step_car_assignment()
car.road_links = car.road_links.drop(columns=[('volume', 'car')]).rename(columns={('all', 'car'): 'volume_car'})
car.road_links['congestion'] = car.road_links['volume_car']/car.road_links['capacity']
car.road_links['volume_bike'] = car.road_links.index.map(bike.road_links['volume_bike'].to_dict())

volume: 100%|██████████| 1/1 [00:00<00:00,  3.90it/s]


### PT assignment

In [73]:
pt = sm.copy()
pt.pt_los = pt.los[~(pt.los['route_type'].isin(['walk', 'car', 'bike']))]
pt.compute_los_volume()

# PT ASSIGNMENT
pt.step_pt_assignment(
    on_road_links=False, boardings=True, alightings=True, transfers=True
)
pt.links['congestion'] = pt.links['load']/pt.links['capacity']

volume: 100%|██████████| 1/1 [00:00<00:00,  3.23it/s]


### Modal split

In [74]:
pt.los.groupby('route_type')['volume'].sum()/pt.los['volume'].sum()

route_type
bike       0.172497
bus        0.015086
car        0.778835
rail       0.000035
tramway    0.033547
Name: volume, dtype: float64

# EXPORT

In [75]:
car.road_links['width_cv'] = width_series(car.road_links['volume_car'])
car.road_links['color_cv'] = color_series(car.road_links['volume_car'], colors=["#210040", "#CFFF20"])
car.road_links['width_bv'] = width_series(car.road_links['volume_bike'])
car.road_links['color_bv'] = color_series(car.road_links['volume_bike'], colors=["#030F0D", "#6AE1FF"])
car.road_links['width_congestion'] = width_series(car.road_links['congestion'])
car.road_links['color_congestion'] = color_series(car.road_links['congestion'], colors=["#FFF7F7", "#B81313"])

pt.links['width_vol'] = width_series(pt.links['load'])
pt.links['color_vol'] = color_series(pt.links['load'], colors=["#210040", "#CFFF20"])
pt.links['width_congestion'] = width_series(pt.links['congestion'])
pt.links['color_congestion'] = color_series(pt.links['width_congestion'], colors=["#FFF7F7", "#B81313"])

In [76]:
car.road_links.drop(columns='index').to_file('../../scenarios/{}/outputs/loaded_road_links.geojson'.format(SCENARIO))
pt.links.to_file('../../scenarios/{}/outputs/pt_loaded_links.geojson'.format(SCENARIO))

In [77]:
end_of_notebook

NameError: name 'end_of_notebook' is not defined

In [ ]:
car.car_los = car.los.loc[car.los['route_type'] == 'car']
car.volumes

### bike = sm.copy()
### bike.car_los = bike.los.loc[bike.los['route_type'] == 'bike']
### bike.step_car_assignment()

,origin,destination,volume
0,zone_00,zone_00,519.720823
1,zone_00,zone_01,556.224143
2,zone_00,zone_02,34.601580
3,zone_00,zone_03,35.038249
4,zone_00,zone_04,61.429419
...,...,...,...
4619,zone_67,zone_63,194.918432
4620,zone_67,zone_64,666.183670
4621,zone_67,zone_65,301.423003
4622,zone_67,zone_66,67.116162


In [ ]:
sm.road_links["capacity"] = sm.road_links['lanes']*sm.road_links['speed']/(sm.road_links['length']/1000) ### TODO Louison : à partir de vitesse max et lanes
sm.zone_to_road["length"] = sm.zone_to_road.geometry.apply(lambda g: g.length)
car = sm.copy()
CAR_OCCUPANCY_RATE = 1.5
# CARS
car_los = sm.los.loc[sm.los['pathfinder_session']=='car_aon']
car_los['volume'] *= 1 / CAR_OCCUPANCY_RATE
car.volumes = car_los[["origin", "destination", "volume"]]

# car.step_road_pathfinder(method='aon', segments=["volume"])
car.step_road_pathfinder(method='aon', segments=["volume"], tolerance=10) # pour les itérations intermédiaires, pas besoin d’un relgap à 0.01%

# TODO Louison: remplacer roadpathfinder MSA par un simple AON + mise à jour des temps sur les road_links à partir des courbes BPR T0*(1+\a * (T/C)**\b)

In [ ]:
sm.pt_los = sm.los[~(sm.los['route_type'].isin(['walk', 'car', 'bike']))]

# PT ASSIGNMENT
sm.step_pt_assignment(
    on_road_links=False, boardings=True, alightings=True, transfers=True
)

# compute TC congested gtime
alpha = 0.15
beta = 4
sm.links["capacity"] = 0.1 
sm.links["time"] = sm.links["time"] * (1 + alpha * sm.links["load"] / sm.links["capacity"]) ** beta

# update pt_los with congested time
d = sm.links["time"].to_dict()
sm.pt_los['in_vehicle_time'] = sm.pt_los['link_path'].apply(lambda ls: sum([d[t] for t in ls]))
cols = [
    'access_time',
    'footpath_time',
    'waiting_time',
    'boarding_time',
    'in_vehicle_time',
]

sm.pt_los['time'] = sm.pt_los[cols].T.sum()

  0%|          | 0/1 [00:00<?, ?it/s]

volume: 100%|██████████| 1/1 [00:01<00:00,  1.86s/it]


# C. CAR ASSIGNMENT

In [ ]:
sm.road_links["capacity"] = 1 ### TODO Louison : à partir de vitesse max et lanes

In [ ]:
sm.zone_to_road["length"] = sm.zone_to_road.geometry.apply(lambda g: g.length)

In [ ]:
car = sm.copy()
CAR_OCCUPANCY_RATE = 1.5  # TODO Louison: valeur de l'occupation moyenne des voitures 
# CARS
car_los = sm.los.loc[sm.los['pathfinder_session']=='car_aon']
car_los['volume'] *= 1 / CAR_OCCUPANCY_RATE
car.volumes = car_los[["origin", "destination", "volume"]]

# car.step_road_pathfinder(method='aon', segments=["volume"])
car.step_road_pathfinder(method='msa', segments=["volume"], tolerance=10) # pour les itérations intermédiaires, pas besoin d’un relgap à 0.01%

# TODO Louison: remplacer roadpathfinder MSA par un simple AON + mise à jour des temps sur les road_links à partir des courbes BPR T0*(1+\a * (T/C)**\b)

# D. MERGE

In [ ]:
#TODO Louison: merge update car los, pt-los, with the rest (bike)
#TODO Louison: back to logit (boucle)

# TOZIPPEDPICKLES

In [ ]:
# TODO: lighten los

In [ ]:
sm.to_zippedpickles(
    mpath + '{scen}/assignment'.format(scen=SCENARIO), 
)

In [ ]:
end_of_notebook

# CONTROL

In [ ]:
temp = sm.los[sm.los['origin']==sm.los['destination']]
round(
    temp.groupby('route_type')[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() /\
    temp[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() * 100
)

In [ ]:
round(
    sm.los.groupby('route_type')[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() /\
    sm.los[['pt_captive', 'car_owners', 'moto_owners']].sum().T.sum() * 100
)

In [ ]:
round(
    sm.los.groupby('route_type')[['pt_captive', 'car_owners', 'moto_owners']].sum() /\
    sm.los[['pt_captive', 'car_owners', 'moto_owners']].sum() * 100
)